# Per-maze Session Chain

Trains one model per maze session by session: every usable session of the maze, all mice, in calendar order. Each session fine-tunes the model carried from the previous one on its first 60%, calibrates on the next 30% and is tested on its last 10%; the first sessions are a warm-up that is never tested. Writes one bundle per seed to `analyses/artifacts/<Paradigm>/ALL_MICE/seed_<SEED>/`, which `cross_maze.ipynb` reads. The compute lives in `src/chain/`; each cell is thin orchestration. Driven per maze and seed by `analyses/run_mazes.py` (`make maze`).

0. Setup: Papermill Parameters And Imports
1. Data: Cohort, Windows, And The Chain
2. The Chain
3. Diagnostics
4. Save The Bundle

## 0. Setup: Papermill Parameters And Imports

In [ ]:
# papermill parameters: run_mazes.py overrides TARGET_PARADIGM, SEED, ARTIFACT_DIR,
# EXCLUDE_SESSIONS and, for smoke runs, the training budgets and QUICK_SWEEP.
# the maze: "More_Water", "Y_Maze" or "Nested_All"
TARGET_PARADIGM = "More_Water"
# restrict to these mouse ids (None = every usable mouse), or hold some out
TARGET_MICE = None
EXCLUDE_MICE = []
# sessions discarded before loading (recording quality), set per maze in run_mazes.py
EXCLUDE_SESSIONS = []
COHORT_LABEL = "ALL_MICE"
# "" derives analyses/artifacts/<Paradigm>/ALL_MICE/seed_<SEED>/
ARTIFACT_DIR = ""
# seed of every trained stage; run_mazes.py sweeps it
SEED = 42
# miscoverage: the coverage target is 1 - ALPHA
ALPHA = 0.10
# per-session training budgets: epoch cap and early-stopping patience
STEP_EPOCHS = 200
EARLY_STOP_PATIENCE = 20
MLP_EPOCHS = 300
MLP_EARLY_STOP_PATIENCE = 30
# the fresh-model-per-session baseline (standard split conformal)
RUN_BASELINE = True
# smoke runs: the coverage sweep at 50% and 90% only
QUICK_SWEEP = False
# a session needs this many calibration and test windows (60/30/10 split)
MIN_VAL_WINDOWS = 50
MIN_TEST_WINDOWS = 20

In [ ]:
# Setup: imports, seeds, device, output paths
import json
import sys
import time
from collections import Counter
from dataclasses import asdict
from pathlib import Path

# run from the repo root so `from src.X import ...` resolves whichever directory
_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists()),
    Path.cwd(),
)
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import torch

from src.chain.order import prepare_chain
from src.chain.runner import SWEEP_TARGETS, ChainConfig, run_chain
from src.chain.stages import TrainBudget
from src.db.client import get_database
from src.utils.artifacts import (
    build_chain_bundle,
    jsonify,
    paradigm_slug,
    sigma_nets_payload,
)
from src.utils.cohort import apply_session_exclusions, discover_cohort
from src.utils.pipeline_prep import (
    apply_window_region_labels,
    assemble_session_data,
    assert_chronological_disjoint,
    force_session_specific,
    gate_sessions_conformal,
)
from src.utils.production_config import (
    CHAIN_SPLIT,
    CHAIN_WARMUP_SESSIONS,
    DROP_STRADDLE_WINDOWS,
    LABEL_REDUCE,
    MLP_DEPTH,
    MLP_HIDDEN,
    WARMUP_TRAIN_FRAC,
)

torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ARTIFACT_DIR = (
    Path(ARTIFACT_DIR)
    if ARTIFACT_DIR
    else _ROOT
    / "analyses"
    / "artifacts"
    / paradigm_slug(TARGET_PARADIGM)
    / COHORT_LABEL
    / f"seed_{SEED}"
)
CHECKPOINT = ARTIFACT_DIR / "checkpoint.pt"
print(f"PyTorch {torch.__version__}  |  device {DEVICE}  |  seed {SEED}")
print(f"bundle -> {ARTIFACT_DIR}")

## 1. Data: Cohort, Windows, And The Chain

Discovers every usable mouse and session in `TARGET_PARADIGM` and pools them into one cohort.

In [ ]:
# Cohort: every usable mouse in TARGET_PARADIGM, pooled into one group. A mouse
# with < 2 valid sessions in the paradigm is skipped.
db = get_database()
print(f"Connected: {db.name}\n")
cohort = discover_cohort(
    db,
    TARGET_PARADIGM,
    target_mice=TARGET_MICE,
    exclude_mice=EXCLUDE_MICE,
)
_PARADIGMS = cohort.paradigms
paradigm = cohort.label
n_classes = cohort.n_classes
group = cohort.group
_pool_pre_exclusion = set(group)
group = apply_session_exclusions(group, EXCLUDE_SESSIONS, log_fn=print)

# provenance for metrics.json so the manual curation is auditable from the saved bundle alone.
EXCLUDED_REQUESTED = list(EXCLUDE_SESSIONS)
EXCLUDED_APPLIED = sorted(_pool_pre_exclusion - set(group))
session_to_mouse = cohort.session_to_mouse
session_to_paradigm = cohort.session_to_paradigm
cohort_mice = cohort.cohort_mice
mouse_id_used = COHORT_LABEL

print(f"cohort {COHORT_LABEL}  paradigm={paradigm!r}  ({n_classes} regions)")
if len(_PARADIGMS) > 1:
    print(f"  pooled over {len(_PARADIGMS)} paradigm blocks: {_PARADIGMS}")
print(f"  {len(cohort_mice)} usable mice: {cohort_mice}")
print(f"  {len(group)} candidate sessions pooled across mice")
if cohort.skipped:
    print(
        f"  {len(cohort.skipped)} mice skipped (no >=2 valid sessions in {TARGET_PARADIGM!r}): "
        f"{[f'M{m}: {w}' for m, w in cohort.skipped]}"
    )

Bins spikes into 1 s windows, labels each window's maze region (Linear: eighths of the run to the trial's water), splits every session chronologically 60/30/10 over task windows, and drops sessions too small to calibrate or test.

In [ ]:
# Assemble: bin spikes -> window -> region labels -> the chronological 60/30/10 split
train_frac, val_frac, test_frac = CHAIN_SPLIT
session_data, valid_sessions, session_configs, _, n_shared = assemble_session_data(
    db, group, paradigm, train_frac=train_frac, val_frac=val_frac, test_frac=test_frac
)
if n_shared > 0:
    force_session_specific(session_configs)
apply_window_region_labels(
    session_data,
    valid_sessions,
    paradigm,
    reduce=LABEL_REDUCE,
    drop_straddle=DROP_STRADDLE_WINDOWS,
    log_fn=print,
)
_unused = [s for s in group if s not in valid_sessions]
if _unused:
    print(f"{len(_unused)} session(s) rejected by assembly: {', '.join(_unused)}")
_dropped = gate_sessions_conformal(
    session_data, valid_sessions, MIN_VAL_WINDOWS, MIN_TEST_WINDOWS
)
for s, nv, nt in _dropped:
    print(f"gated out {s}: calibration {nv}, test {nt}")
assert_chronological_disjoint(session_data, valid_sessions)
if len(valid_sessions) <= CHAIN_WARMUP_SESSIONS:
    raise RuntimeError(
        f"{paradigm}: {len(valid_sessions)} sessions, nothing to test after the warm-up"
    )
print(f"{len(valid_sessions)} usable sessions")

Orders the sessions into the chain (calendar order, all mice), re-splits the warm-up sessions 70/30, and z-scores every session's neurons on its own training block.

In [ ]:
# The chain: calendar order, warm-up re-split, per-session normalization
chain = prepare_chain(
    session_data, valid_sessions, CHAIN_WARMUP_SESSIONS, WARMUP_TRAIN_FRAC
)
mouse = {s: str(session_to_mouse[s]) for s in chain}
print(
    f"chain: {len(chain)} sessions from {len(set(mouse.values()))} mice  |  "
    f"warm-up {chain[:CHAIN_WARMUP_SESSIONS]}  |  {len(chain) - CHAIN_WARMUP_SESSIONS} tested"
)
for m, n in sorted(Counter(mouse.values()).items()):
    print(f"  M{m}: {n} sessions")

Region distribution over every task window of the chain, against the chance and majority-region baselines.

In [ ]:
_counts = np.zeros(n_classes)
for s in chain:
    d = session_data[s]
    _task = np.concatenate([d["train_idx"], d["val_idx"], d["test_idx"]])
    _counts += np.bincount(d["regions"][_task], minlength=n_classes)
_share = _counts / _counts.sum()
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.bar(range(n_classes), _share, color="#4a7fb5")
ax.axhline(1 / n_classes, ls="--", color="k", lw=1, label=f"chance {1 / n_classes:.1%}")
ax.set_xticks(range(n_classes))
ax.set_xticklabels([f"R{r}" for r in range(n_classes)])
ax.set_ylabel("fraction of task windows")
ax.set_title(f"Region distribution, {paradigm}")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 2. The Chain

Runs the chain: each session fine-tunes the encoder, decoder and sigma carried from the previous session, and every session after the warm-up is evaluated: the method, the pool-rule sensitivity, the sigma = 1 ablation, the coverage sweep and the fresh-model baseline. A checkpoint after every session lets an interrupted run resume where it stopped.

In [ ]:
cfg = ChainConfig(
    seed=SEED,
    device=DEVICE,
    alpha=ALPHA,
    sweep_targets=(0.5, 0.9) if QUICK_SWEEP else SWEEP_TARGETS,
    baseline=RUN_BASELINE,
    budget=TrainBudget(
        STEP_EPOCHS, EARLY_STOP_PATIENCE, MLP_EPOCHS, MLP_EARLY_STOP_PATIENCE
    ),
)
_t0 = time.time()
rows = run_chain(
    session_data, chain, session_configs, n_classes, mouse, cfg, checkpoint=CHECKPOINT
)
print(f"chain done in {(time.time() - _t0) / 60:.1f} min, {len(rows)} sessions tested")

## 3. Diagnostics

Every method pooled over the test windows of all tested sessions, then per session in chain order (grey lines: a mouse's first session).

In [ ]:
metrics, arrays = build_chain_bundle(
    rows,
    chain=chain,
    n_warmup=CHAIN_WARMUP_SESSIONS,
    session_data=session_data,
    n_classes=n_classes,
    paradigm=paradigm,
    seed=SEED,
    alpha=ALPHA,
    rule=cfg.rule,
    config=asdict(cfg),
)
metrics["excluded_sessions"] = {
    "requested": EXCLUDED_REQUESTED,
    "applied": EXCLUDED_APPLIED,
}
print(
    f"  {'method':<14} {'cov':>6} {'|C|/K':>6} {'acc':>6} {'|gap|':>6} {'below':>6} {'global':>7}"
)
for name, m in metrics["conformal"]["methods"]["all"].items():
    print(
        f"  {name:<14} {m['coverage']:>6.3f} {m['size_over_k']:>6.3f} {m['accuracy']:>6.3f} "
        f"{m['mean_abs_gap']:>6.3f} {m['sessions_below']:>6d} {m['share_global']:>7.2f}"
    )

_x = np.arange(len(rows))
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
for name, style in ((f"chain_{cfg.rule}", "o-"), ("fresh", "s-")):
    if name not in rows[0]["methods"]:
        continue
    ax[0].plot(
        _x, [1 - r["methods"][name]["error"].mean() for r in rows], style, label=name
    )
    ax[1].plot(
        _x, [r["methods"][name]["covered"].mean() for r in rows], style, label=name
    )
for a in ax:
    for i, r in enumerate(rows):
        if r["first_of_mouse"]:
            a.axvline(i, color="0.85", lw=1, zorder=0)
    a.set_xlabel("tested session, chain order")
    a.grid(alpha=0.3)
ax[0].set_ylabel("accuracy")
ax[1].set_ylabel("coverage")
ax[1].axhline(1 - ALPHA, ls="--", color="k", lw=1)
ax[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

## 4. Save The Bundle

Writes `sigma_nets.pt`, `metrics.json` and, last, `arrays.npz` (its presence marks the bundle complete for `run_mazes.py`), then removes the chain checkpoint.

In [ ]:
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
torch.save(
    sigma_nets_payload(rows, MLP_HIDDEN, MLP_DEPTH), ARTIFACT_DIR / "sigma_nets.pt"
)
(ARTIFACT_DIR / "metrics.json").write_text(json.dumps(jsonify(metrics), indent=2))
np.savez_compressed(ARTIFACT_DIR / "arrays.npz", **arrays)
CHECKPOINT.unlink(missing_ok=True)
print(f"saved -> {ARTIFACT_DIR}: sigma_nets.pt, metrics.json, arrays.npz")